## Preparing the dataset for modelling (NOT READY YET)

### general paths and libraries

In [ ]:
!pip install seaborn

In [ ]:
import os
import numpy as np
import rasterio
from rasterio import mask
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
# variables - Best results (pearson correlation with LST) for window size
lst_clipped_path = '../../datos-notebooks/0.LST/LST_clipped_celsius.tif'
ndwi_1_path = '../../datos-notebooks/1.land-cover/moving-windows/ndwi/Class_1_window61.tif'
ndvi_1_path = '../../datos-notebooks/1.land-cover/moving-windows/ndvi/Class_1_window25.tif'
veg_vol_path = '../../datos-notebooks/2.objects-height/moving-windows/tree-volume/trees_window13.tif'
built_vol_path = '../../datos-notebooks/2.objects-height/moving-windows/building-volume/buildings_window13.tif'
built_area_path = '../../datos-notebooks/2.objects-height/moving-windows/building-area/building_area_window11.tif'
building_height_path = '../../datos-notebooks/2.objects-height/moving-windows/building-height/building_height_window21.tif'
mean_elev_path = '../../datos-notebooks/3.topography/moving_windows/elev_mean/elev_mean_win111.tif'
mean_slope_path = '../../datos-notebooks/3.topography/moving_windows/slope_mean/slope_mean_win19.tif'
northness_path = '../../datos-notebooks/3.topography/moving_windows/northness/northness_win201.tif'

output_dir = '../../datos-notebooks/4.LST-prediction/'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
with rasterio.open(lst_clipped_path) as src:
    lst = src.read(1)

lst_mask = np.isnan(lst)

# mask values out of city boundary
ndwi_1 = np.where(lst_mask, np.nan, rasterio.open(ndwi_1_path).read(1).astype("float32"))
ndvi_1 = np.where(lst_mask, np.nan, rasterio.open(ndvi_1_path).read(1).astype("float32"))
veg_vol = np.where(lst_mask, np.nan, rasterio.open(veg_vol_path).read(1).astype("float32"))
built_vol = np.where(lst_mask, np.nan, rasterio.open(built_vol_path).read(1).astype("float32"))
built_area = np.where(lst_mask, np.nan, rasterio.open(built_area_path).read(1).astype("float32"))
built_height = np.where(lst_mask, np.nan, rasterio.open(building_height_path).read(1).astype("float32"))
mean_elev = np.where(lst_mask, np.nan, rasterio.open(mean_elev_path).read(1).astype("float32"))
mean_slope = np.where(lst_mask, np.nan, rasterio.open(mean_slope_path).read(1).astype("float32"))
northness = np.where(lst_mask, np.nan, rasterio.open(northness_path).read(1).astype("float32"))

rasters_masked = {
    "Water": ndwi_1,
    "Dense vegetation presence": ndvi_1,
    "Vegetation volume": veg_vol,
    "Built volume": built_vol,
    "Built area": built_area,
    "Building height": built_height,
    "Elevation": mean_elev,
    "Slope": mean_slope,
    "Northness": northness
}


In [ ]:
#plot masked rasters 

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, data) in zip(axes.flat, rasters_masked.items()):

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:
# Plot histograms of masked rasters
fig, axes = plt.subplots(3, 3, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, arr) in enumerate(rasters_masked.items()):
    # Plot histogram using already-masked arrays
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")

# Hide unused subplots (if less than 8 rasters)
for j in range(i + 1, 8):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


### standarize variables - mean:0, std:1

In [ ]:
output_dir = '../../datos-notebooks/4.LST-prediction/standarized_variables'
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Z-score standardization for regression predictors

def standardize_array(arr, profile, out_path):
    mean = np.nanmean(arr)
    std = np.nanstd(arr)
    z = (arr - mean) / std

    profile.update(dtype='float32')

    with rasterio.open(out_path, 'w', **profile) as dst:
        dst.write(z.astype('float32'), 1)

In [ ]:
# Apply standarization
with rasterio.open(lst_clipped_path) as src:
    profile = src.profile.copy()
    
standardize_array(ndwi_1, profile, os.path.join(output_dir, 'NDWI_1_z.tif'))
standardize_array(ndvi_1, profile, os.path.join(output_dir, 'NDVI_1_z.tif'))
standardize_array(veg_vol, profile, os.path.join(output_dir, 'VEG_VOL_z.tif'))
standardize_array(built_vol, profile, os.path.join(output_dir, 'BUILT_VOL_z.tif'))
standardize_array(built_area, profile, os.path.join(output_dir, 'BUILT_AREA_z.tif'))
standardize_array(built_height, profile, os.path.join(output_dir, 'BUILDING_HEIGHT_z.tif'))
standardize_array(mean_elev, profile, os.path.join(output_dir, 'ELEV_z.tif'))
standardize_array(mean_slope, profile, os.path.join(output_dir, 'SLOPE_z.tif'))
standardize_array(northness, profile, os.path.join(output_dir, 'NORTH_z.tif'))

In [ ]:
std_rasters_paths = {
    "Water": os.path.join(output_dir, "NDWI_1_z.tif"),
    "Dense veg. presence": os.path.join(output_dir, "NDVI_1_z.tif"),
    "Vegetation Volume": os.path.join(output_dir, "VEG_VOL_z.tif"),
    "Built Volume": os.path.join(output_dir, "BUILT_VOL_z.tif"),
    "Built Area": os.path.join(output_dir, "BUILT_AREA_z.tif"),
    "Built Height": os.path.join(output_dir, "BUILDING_HEIGHT_z.tif"),
    "Elevation": os.path.join(output_dir, "ELEV_z.tif"),
    "Slope": os.path.join(output_dir, "SLOPE_z.tif"),
    "Northness": os.path.join(output_dir, "NORTH_z.tif")
}

In [ ]:
#plot standardized rasters

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for ax, (name, path) in zip(axes.flat, std_rasters_paths.items()):
    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

    img = ax.imshow(data, cmap='viridis')
    ax.set_title(name)
    ax.axis('off')
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)


plt.tight_layout()
plt.show()

In [ ]:

fig, axes = plt.subplots(3, 3, figsize=(16, 8))
axes = axes.flatten()  # Flatten to 1D for easy indexing

for i, (name, path) in enumerate(std_rasters_paths.items()):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        nodata = src.nodata

    # Plot histogram
    axes[i].hist(arr[~np.isnan(arr)], bins=50)
    axes[i].set_title(name)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Frequency")
    axes[i].axvline(0, linestyle='--')

plt.tight_layout()
plt.show()

### stratified sampling of LST

In [ ]:
rasters = {"LST": lst_clipped_path} # create dictionary with LST raster (not standardized)
rasters.update(std_rasters_paths) # add standardized rasters to the dictionary
#rasters.pop("Elevation", None)
rasters.pop("Built Volume", None)

In [ ]:
rasters

In [ ]:
n_bins = 10

In [ ]:
# Read and flatten
data_arrays = {}
for name, path in rasters.items():
    with rasterio.open(path) as src:
        arr = src.read(1)
        data_arrays[name] = arr.flatten()

df = pd.DataFrame(data_arrays)

# Drop rows with NaN
df = df.dropna()

# Create LST bins for stratification (5 bins)

df['LST_bin'] = pd.qcut(df['LST'], n_bins, labels=False)


In [ ]:
# Stratified sampling: 10% of each bin
sample_fraction = 0.1
df_sampled = df.groupby('LST_bin', group_keys=False).apply(
    lambda x: x.sample(frac=sample_fraction, random_state=42)
)

# Drop bin column
df_sampled = df_sampled.drop(columns=['LST_bin']).reset_index(drop=True)

print(df_sampled.head())
print("Sampled dataset shape:", df_sampled.shape)


# Split features and target
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']



In [ ]:
# Recreate bins for the sampled dataset
df_sampled['LST_bin'] = pd.qcut(df_sampled['LST'], n_bins, labels=False)

# Show min and max LST in each bin
for b in range(n_bins):
    values_in_bin = df_sampled[df_sampled['LST_bin'] == b]['LST']
    print(f"Bin {b}: min = {values_in_bin.min():.2f}, max = {values_in_bin.max():.2f}, count = {len(values_in_bin)}")



In [ ]:
# Compute correlation matrix (only features)
corr_matrix = X.corr()

# Display heatmap
plt.figure(figsize=(8,6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# --- Vegetation plot ---
plt.figure()
plt.scatter(
    df_sampled["Dense veg. presence"],
    df_sampled["Vegetation Volume"],
    s=1, alpha=0.5
)
plt.xlabel("Vegetation presence (z)")
plt.ylabel("Vegetation volume (z)")
plt.title("Vegetation: Area vs Volume")
plt.show()

# --- Built plot ---
plt.figure()
plt.scatter(
    df_sampled["Built Area"],
    df_sampled["Built Height"],
    s=1, alpha=0.5
)
plt.xlabel("Built area (z)")
plt.ylabel("Built Height (z)")
plt.title("Built: Area vs Height")
plt.show()


### Descriptive statistics

In [ ]:
lst_vals = df_sampled["LST"]
predictors = [v for v in df_sampled.columns if v != "LST"]
n_vars = len(predictors)


fig, axes = plt.subplots(3, 3, figsize=(10, 12))
axes = axes.flatten()  # para indexar más fácil

for i, v in enumerate(predictors):
    axes[i].scatter(df_sampled[v], lst_vals, s=1, alpha=0.3)
    axes[i].set_xlabel(v)
    axes[i].set_ylabel("LST")
    axes[i].set_title(f"LST vs {v}")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for i, v in enumerate(df.columns):
    axes[i].boxplot(df[v], vert=False)
    axes[i].set_title(v)

for ax in axes[len(df.columns):]:
    ax.axis('off')

plt.tight_layout()
plt.show()


### PCA

In [ ]:
df_sampled = df_sampled.drop(columns=['LST_bin'])

In [ ]:
df_sampled

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


y = df_sampled["LST"].values
X = df_sampled.drop(columns=["LST"])


In [ ]:
X

In [ ]:

predictor_names = X.columns.tolist()

# Standardization
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# PCA
pca = PCA()
X_pca = pca.fit_transform(X_std)

# ---------------------------
# Explained variance
# ---------------------------
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)

print("\nCumulative variance:")
print(np.cumsum(pca.explained_variance_ratio_))




In [ ]:
import pandas as pd
import numpy as np

# Loadings: correlación de cada variable con cada PC
loadings = pd.DataFrame(
    pca.components_.T,       # cada columna es un PC
    columns=[f"PC{i+1}" for i in range(X.shape[1])],
    index=predictor_names     # nombres de variables originales
)

print(loadings)


In [ ]:
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ---------------------------
# Datos
# ---------------------------
X = df_sampled[selected_features]

# ---------------------------
# Estandarización
# ---------------------------
scaler = StandardScaler()
X_std = scaler.fit_transform(X)

# ---------------------------
# PCA
# ---------------------------
pca = PCA()
X_pca = pca.fit_transform(X_std)

# ---------------------------
# Crear DataFrame de loadings
# ---------------------------
loadings = pd.DataFrame(
    pca.components_.T,        # componentes
    index=selected_features,   # nombres de variables
    columns=[f"PC{i+1}" for i in range(len(selected_features))]
)

# ---------------------------
# Heatmap
# ---------------------------
plt.figure(figsize=(12,6))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0)
plt.title("PC Loadings")
plt.show()


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(np.arange(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1)
plt.title("Cumulative variance")
plt.grid(True)
plt.show()


### linear model using PCs

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

# ---------------------------
# Choose how many PCs to use
# ---------------------------
X = X_pca[:, :7]  # first 6 PCs
y = df_sampled["LST"].values

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# ---------------------------
# Add intercept for statsmodels
# ---------------------------
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# ---------------------------
# Fit OLS model
# ---------------------------
ols_model = sm.OLS(y_train, X_train_sm).fit()

# ---------------------------
# Print summary
# ---------------------------
print(ols_model.summary())

# ---------------------------
# Predict on test set
# ---------------------------
y_pred = ols_model.predict(X_test_sm)

# ---------------------------
# Evaluation
# ---------------------------
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")



### linear model using PCs stepwise (AIC based)

In [ ]:
!pip install statsmodels
import numpy as np
import statsmodels.api as sm

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# ---------------------------
# Select PCs
# ---------------------------
X = X_pca[:, :7]  # first 6 PCs
y = df_sampled["LST"].values

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# ---------------------------
# Add intercept
# ---------------------------
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

# ---------------------------
# Stepwise forward selection (train only)
# ---------------------------
pc_names = [f"PC{i+1}" for i in range(X_train.shape[1])]
selected = []
remaining = pc_names.copy()
current_aic = np.inf
min_delta = 100   # minimum AIC improvement

while remaining:
    aic_candidates = []
    for candidate in remaining:
        idx = [pc_names.index(v) for v in selected + [candidate]]
        X_tmp = X_train_sm[:, [0] + [i+1 for i in idx]]  # intercept + PCs
        model = sm.OLS(y_train, X_tmp).fit()
        aic_candidates.append((model.aic, candidate))
    
    aic_candidates.sort()
    best_aic, best_candidate = aic_candidates[0]
    
    if current_aic - best_aic > min_delta:
        current_aic = best_aic
        selected.append(best_candidate)
        remaining.remove(best_candidate)
        print(f"Added: {best_candidate}, AIC = {best_aic:.2f}")
    else:
        print("No candidate improved AIC enough. Stopping.")
        break

print("\nSelected PCs (train):", selected)

# ---------------------------
# Fit final model on train with selected PCs
# ---------------------------
final_idx = [pc_names.index(v) for v in selected]
X_train_final = X_train_sm[:, [0] + [i+1 for i in final_idx]]
X_test_final  = X_test_sm[:, [0] + [i+1 for i in final_idx]]

final_model = sm.OLS(y_train, X_train_final).fit()
print("\n--- Train summary ---")
print(final_model.summary())

# ---------------------------
# Predict on test set
# ---------------------------
y_pred = final_model.predict(X_test_final)

# ---------------------------
# Evaluate test set
# ---------------------------
r2_test = r2_score(y_test, y_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest set R²: {r2_test:.3f}")
print(f"Test set RMSE: {rmse_test:.3f}")



In [ ]:
# not updated
'''
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

# Crear DataFrame de loadings
loadings = {
    "PC1": [0.028, 0.452, 0.262, -0.335, -0.348, 0.403, 0.420, 0.392],
    "PC2": [-0.150, 0.089, 0.408, 0.536, 0.510, 0.340, 0.336, -0.164],
    "PC3": [0.936, -0.029, 0.207, 0.132, 0.127, -0.076, -0.058, 0.195],
    "PC4": [-0.033, 0.091, 0.728, -0.214, -0.225, -0.336, -0.136, -0.482],
    "PC5": [-0.309, 0.054, 0.337, 0.091, 0.210, -0.285, -0.389, 0.709],
    "PC6": [0.050, 0.878, -0.270, 0.174, 0.114, -0.243, -0.164, -0.160],
    "PC7": [-0.025, -0.051, -0.072, -0.158, 0.232, -0.648, 0.695, 0.100],
    "PC8": [-0.037, -0.056, 0.004, 0.688, -0.664, -0.209, 0.164, 0.101]
}

variables = ["Water", "Dense veg.", "Veg Vol", "Built Vol", "Built Area", "Elev", "Slope", "Northness"]

df_loadings = pd.DataFrame(loadings, index=variables)

plt.figure(figsize=(10,6))
sns.heatmap(df_loadings, annot=True, cmap="coolwarm", center=0)
plt.title("PC Loadings")
plt.show()
'''

### linear regression with variables

In [ ]:
# Split features and target from the sampled DataFrame
X = df_sampled.drop(columns=['LST'])
y = df_sampled['LST']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# ---------------------------
# Add intercept (constant) for OLS
# ---------------------------
X_train_sm = sm.add_constant(X_train)
X_test_sm  = sm.add_constant(X_test)

# ---------------------------
# Fit OLS model on training set
# ---------------------------
ols_model = sm.OLS(y_train, X_train_sm).fit()

# ---------------------------
# Print summary (train performance)
# ---------------------------
print(ols_model.summary())

# ---------------------------
# Predict on test set
# ---------------------------
y_pred = ols_model.predict(X_test_sm)

# ---------------------------
# Evaluate test set
# ---------------------------
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")




In [ ]:
# with stepwise

import statsmodels.api as sm

def stepwise_selection(X, y, 
                       initial_list=[], 
                       threshold_in=0.01, 
                       threshold_out=0.05, 
                       verbose=True):
    """
    Forward-backward feature selection using p-values from statsmodels.api.OLS
    """
    included = list(initial_list)
    while True:
        changed=False
        # forward step
        excluded = list(set(X.columns) - set(included))
        new_pval = pd.Series(index=excluded)
        for new_column in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [new_column]]))).fit()
            new_pval[new_column] = model.pvalues[new_column]
        if not new_pval.empty:
            best_pval = new_pval.min()
            if best_pval < threshold_in:
                best_feature = new_pval.idxmin()
                included.append(best_feature)
                changed=True
                if verbose:
                    print('Add  {:30} with p-value {:.6f}'.format(best_feature, best_pval))
        
        # backward step
        model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
        # use all coefs except intercept
        pvalues = model.pvalues.iloc[1:]
        worst_pval = pvalues.max()  # null if pvalues is empty
        if worst_pval > threshold_out:
            changed=True
            worst_feature = pvalues.idxmax()
            included.remove(worst_feature)
            if verbose:
                print('Drop {:30} with p-value {:.6f}'.format(worst_feature, worst_pval))
        if not changed:
            break
    return included

# Run stepwise selection
selected_features = stepwise_selection(X, y)

print("Selected features:", selected_features)



In [ ]:
# Fit final OLS
X_final = sm.add_constant(X[selected_features])
ols_model = sm.OLS(y, X_final).fit()
print(ols_model.summary())

# Predicted values
y_pred_ols = ols_model.predict(X_final)


In [ ]:
import statsmodels.api as sm
import pandas as pd

def stepwise_selection_aic(X, y, initial_list=[], min_delta=0.0, verbose=True):
    """
    Forward-backward stepwise selection based on AIC.
    
    Parameters:
        X : pandas DataFrame with predictors
        y : target vector
        initial_list : list of variables to start with
        min_delta : minimum AIC improvement to accept a change
        verbose : whether to print steps
        
    Returns:
        included : list of selected features
    """
    included = list(initial_list)
    current_aic = None

    while True:
        changed = False
        # forward step
        excluded = list(set(X.columns) - set(included))
        aic_with_candidates = []
        for col in excluded:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included + [col]]))).fit()
            aic_with_candidates.append((model.aic, col))
        if aic_with_candidates:
            aic_with_candidates.sort()
            best_aic, best_candidate = aic_with_candidates[0]
            if current_aic is None or (current_aic - best_aic) > min_delta:
                included.append(best_candidate)
                current_aic = best_aic
                changed = True
                if verbose:
                    print(f"Add {best_candidate:30} AIC = {best_aic:.2f}")

        # backward step
        if included:
            model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[included]))).fit()
            aic_without_candidates = []
            for col in included:
                temp = included.copy()
                temp.remove(col)
                if temp:
                    temp_model = sm.OLS(y, sm.add_constant(pd.DataFrame(X[temp]))).fit()
                    aic_without_candidates.append((temp_model.aic, col))
            if aic_without_candidates:
                aic_without_candidates.sort()
                best_aic_back, worst_candidate = aic_without_candidates[0]
                if current_aic - best_aic_back > min_delta:
                    included.remove(worst_candidate)
                    current_aic = best_aic_back
                    changed = True
                    if verbose:
                        print(f"Drop {worst_candidate:30} AIC = {best_aic_back:.2f}")

        if not changed:
            break

    return included

# ---------------------------
# Example usage
# ---------------------------
selected_features = stepwise_selection_aic(X, y)
print("Selected features:", selected_features)


In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np


# ---------------------------
# Crear X e y a partir del df_sampled
# ---------------------------
X = df_sampled[selected_features]
y = df_sampled["LST"].values

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# Agregar constante (intercepto)
# ---------------------------
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# ---------------------------
# Ajustar OLS
# ---------------------------
ols_model = sm.OLS(y_train, X_train_const).fit()

# ---------------------------
# Mostrar summary del modelo
# ---------------------------
print("--- Train summary ---")
print(ols_model.summary())

# ---------------------------
# Predecir sobre test set
# ---------------------------
y_pred = ols_model.predict(X_test_const)

# ---------------------------
# Evaluación sobre test set
# ---------------------------
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")


### RF

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# ---------------------------
# Variables seleccionadas
# ---------------------------
X = df_sampled[selected_features]
y = df_sampled["LST"].values

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---------------------------
# Random Forest Regressor
# ---------------------------
rf_model = RandomForestRegressor(
    n_estimators=500,       # number of trees
    max_depth=None,         # no max depth
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# ---------------------------
# Predicciones y evaluación
# ---------------------------
y_pred = rf_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R²: {r2:.3f}")
print(f"RMSE: {rmse:.3f}")

# ---------------------------
# Importancia de variables
# ---------------------------
importances = rf_model.feature_importances_
for f, imp in zip(selected_features, importances):
    print(f"{f}: {imp:.3f}")


#### OLS with polynomial terms in original variables

In [ ]:
import statsmodels.api as sm
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# ---------------------------
# Variables seleccionadas
# ---------------------------
X = df_sampled[selected_features]
y = df_sampled["LST"].values

# ---------------------------
# Crear polinomios (grado 2)
# ---------------------------
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(X)
poly_features = poly.get_feature_names_out(selected_features)

X_poly_df = pd.DataFrame(X_poly, columns=poly_features)

# ---------------------------
# Train-test split
# ---------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_poly_df, y, test_size=0.2, random_state=42
)

# ---------------------------
# Stepwise selection using AIC
# ---------------------------
def stepwise_selection_aic(X, y, verbose=True):
    included = []
    best_aic = np.inf

    while True:
        changed = False

        # Forward step
        excluded = list(set(X.columns) - set(included))
        aic_with_candidates = {}
        for col in excluded:
            model = sm.OLS(y, sm.add_constant(X[included + [col]])).fit()
            aic_with_candidates[col] = model.aic
        if aic_with_candidates:
            best_candidate = min(aic_with_candidates, key=aic_with_candidates.get)
            if aic_with_candidates[best_candidate] < best_aic:
                included.append(best_candidate)
                best_aic = aic_with_candidates[best_candidate]
                changed = True
                if verbose:
                    print(f"Add {best_candidate:40} AIC = {best_aic:.2f}")

        # Backward step
        if included:
            aic_without_candidates = {}
            for col in included:
                cols = included.copy()
                cols.remove(col)
                if cols:
                    model = sm.OLS(y, sm.add_constant(X[cols])).fit()
                    aic_without_candidates[col] = model.aic
                else:
                    aic_without_candidates[col] = np.inf
            worst_candidate = min(aic_without_candidates, key=aic_without_candidates.get)
            if aic_without_candidates[worst_candidate] < best_aic:
                included.remove(worst_candidate)
                best_aic = aic_without_candidates[worst_candidate]
                changed = True
                if verbose:
                    print(f"Drop {worst_candidate:40} AIC = {best_aic:.2f}")

        if not changed:
            break

    return included

selected_poly_features = stepwise_selection_aic(X_train, y_train)
print("\nSelected polynomial features (AIC):", selected_poly_features)




In [ ]:
# ---------------------------
# Ajustar OLS final con las seleccionadas
# ---------------------------
X_train_sel = sm.add_constant(X_train[selected_poly_features])
X_test_sel = sm.add_constant(X_test[selected_poly_features])

ols_poly_model = sm.OLS(y_train, X_train_sel).fit()
print(ols_poly_model.summary())

# ---------------------------
# Predicción y evaluación
# ---------------------------
y_pred = ols_poly_model.predict(X_test_sel)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nTest set R²: {r2:.3f}")
print(f"Test set RMSE: {rmse:.3f}")

